In [ ]:

using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120
const aa = 40
const N = 1_000_000
const I0 = 10
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)

data_file_for(tag::Symbol) =
    tag === :memoryless  ? "data/memoryless.csv" :
    tag === :sliding     ? "data/sliding_kmax14.csv" :          
    tag === :powerlaw    ? "data/powerlaw_lambdaP.csv" :
    tag === :exponential ? "data/exponential_lambdaE.csv" :
    tag === :reciprocal  ? "data/reciprocal_lambdaR.csv" :     
    error("Unknown data tag: $tag")

model_tag_sym = :powerlaw
data_tag_sym  = :powerlaw

KMAX_UPPER = 30  

data_path = data_file_for(data_tag_sym)
raw, hdr = DelimitedFiles.readdlm(data_path, ',', header=true)
raw = Matrix{Float64}(raw)
tau = size(raw, 2) ÷ 3

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

i = 13
c = 3
Istar_obs = Vector{Int}(round.(Int, raw[i, 1:tau]))
@info "[$(String(model_tag_sym))_model_$(String(data_tag_sym))_data] Fitting dataset $(i) chain $(c) (tau=$tau)"

Random.seed!(2025 + i * 100 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_sim_$(i)_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_sim_$(i)_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))

    el = Dates.value(Dates.now() - t0) / 1000
    @info(@sprintf("Dataset %03d chain %d: ok in %.2fs", i, c, el))
catch err
    el = Dates.value(Dates.now() - t0) / 1000
    @warn(@sprintf("Dataset %03d chain %d: ERROR - %s after %.2fs", i, c, err))
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [powerlaw_model_powerlaw_data] Fitting dataset 13 chain 3 (tau=50)
[ Info: [powerlaw] iter 1000/1000000 elapsed=3.5s, rate=0.320, medians=[0.776, 0.00461, 0.315, 0.599], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [powerlaw] iter 2000/1000000 elapsed=7.4s, rate=0.306, medians=[0.766, 0.00459, 0.306, 0.594], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [powerlaw] iter 3000/1000000 elapsed=10.6s, rate=0.304, medians=[0.767, 0.00449, 0.300, 0.596], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [powerlaw] iter 4000/1000000 elapsed=13.8s, rate=0.303, medians=[0.770, 0.00446, 0.300, 0.599], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [powerlaw] iter 5000/1000000 elapsed=17.0s, rate=0.308, medians=[0.771, 0.00449, 0.304, 0.599], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [powerlaw] iter 6000/1000000 elapsed=20.2s, rate=0.314, medians=[0.778, 0.00448, 0.310, 0.605], std=[0.0035, 0.000035, 0.0035, 0.0035] [ADAPT]
[ Info: [powerlaw] it